In [1]:
# Import
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

In [2]:
# Load iris dataset
from sklearn.datasets import load_iris

# Instantiate
iris = load_iris()

# Create training and feature
X = iris.data
y = iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

In [3]:
# Same 3-step process

# 1. Instantiate
# default criterion=gini
# swap to criterion=entropy 
dtc = DecisionTreeClassifier(random_state=0, criterion='gini')

# 2. Fit
dtc.fit(X_train, y_train)

# 3. Predict, there're 4 features in the iris dataset
y_pred_class = dtc.predict(X_test)


### Parameters of DecisionTreeClassifier

- **criterion**  
  Function to measure split quality.  
  - `"gini"` (default): Uses *Gini impurity*.  
  - `"entropy"`: Uses *Information Gain*.  

- **random_state**  
  Controls randomness. Ensures reproducible results.  

- **max_depth**  
  Maximum depth of the tree. Prevents overfitting.  

- **min_samples_split**  
  Minimum number of samples required to split an internal node.  

- **min_samples_leaf**  
  Minimum number of samples required to be at a leaf node.  

- **max_features**  
  Number of features to consider when looking for the best split.  


## Evaluating with accuracy

In [4]:
from sklearn import metrics
# Accuracy
metrics.accuracy_score(y_test, y_pred_class)

0.9736842105263158

## Dealing with overfitting: min_samples_split

default min_samples_split=2
* This means if you've 1 sample left in a node, you cannot split further
* When it gets really deep in depth, it overfits your data

If you increase your min_samples_split value
* You would decrease the depth of your tree
* This is because you would run out of samples to split
* This would reduce overfitting

In [5]:
# 1. Instantiate with min_samples_split = 50
dtc = DecisionTreeClassifier(min_samples_split=50, random_state=0)

# 2. Fit
dtc.fit(X_train, y_train)

# 3. Predict, there're 4 features in the iris dataset
y_pred_class = dtc.predict(X_test)

# Accuracy
metrics.accuracy_score(y_test, y_pred_class)

0.8947368421052632

### Influence of DecisionTreeClassifier Parameters

- **criterion**
  - Determines how the tree chooses the best split at each node.
  - `"gini"` → prefers splits that reduce class mixing (impurity).
  - `"entropy"` → prefers splits that maximize information gain.
  - Influence: Affects the *shape* of the tree and which features are chosen.

- **random_state**
  - Controls the randomness in:
    - Feature selection (when `max_features < n_features`).
    - Tie-breaking when multiple splits are equally good.
  - Influence: Ensures reproducibility. Same seed = same tree.

- **max_depth**
  - Limits how many levels the tree can grow.
  - Small value → shallow tree, less complex, less overfitting.
  - Large value (or `None`) → tree grows until all leaves are pure, higher risk of overfitting.
  - Influence: Directly controls complexity and generalization.

- **min_samples_split**
  - Minimum number of samples required at a node before splitting.
  - Higher value → fewer splits, simpler tree.
  - Lower value → more splits, deeper tree.
  - Influence: Balances complexity vs. simplicity.

- **min_samples_leaf**
  - Minimum number of samples required at each leaf.
  - Ensures leaves are not too small.
  - Higher value → smoother predictions (more generalized).
  - Influence: Prevents overfitting to small sample sets.

- **max_features**
  - Number of features to consider when choosing the best split.
  - `"auto"` / `None` → all features.
  - `"sqrt"` → square root of features (often used in classification).
  - `"log2"` → log base 2 of features.
  - Influence: Introduces randomness, reduces correlation between trees (especially useful in ensembles like Random Forests).

---
### Example
```python
dtc = DecisionTreeClassifier(
    criterion="entropy",      # Split using information gain
    random_state=42,          # Reproducible results
    max_depth=5,              # Prevent overfitting by limiting depth
    min_samples_split=10,     # Nodes must have at least 10 samples to split
    min_samples_leaf=5,       # Leaves must have at least 5 samples
    max_features="sqrt"       # Each split considers only sqrt(n_features)
)


## Using GridSearchCV for optimizing parameters
* Hyper-parameters are parameters that are not directly learnt within estimators. In scikit-learn they are passed as arguments to the constructor of the estimator classes. 

* It is possible and recommended to search the hyper-parameter space for the best cross validation score.

* Note that it is common that a small subset of those parameters can have a large impact on the predictive or computation performance of the model while others can be left to their default values.

In [6]:
# Import
from sklearn.model_selection import GridSearchCV

# Define the parameter values that should be searched
sample_split_range = list(range(2, 50))

# Create a parameter grid: map the parameter names to the values that should be searched
# Simply a python dictionary
# Key: parameter name
# Value: list of values that should be searched for that parameter
# Single key-value pair for param_grid
#param_grid = dict(min_samples_split=sample_split_range)

param_grid = {'min_samples_split': list(range(2, 50)), 'criterion': ['gini', 'entropy']}

# instantiate the grid
grid = GridSearchCV(dtc, param_grid, cv=10, scoring='accuracy')

# fit the grid with data
grid.fit(X_train, y_train)

GridSearchCV(cv=10,
             estimator=DecisionTreeClassifier(min_samples_split=50,
                                              random_state=0),
             param_grid={'criterion': ['gini', 'entropy'],
                         'min_samples_split': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
                                               12, 13, 14, 15, 16, 17, 18, 19,
                                               20, 21, 22, 23, 24, 25, 26, 27,
                                               28, 29, 30, 31, ...]},
             scoring='accuracy')

A search consists of:
* an estimator (regressor or classifier )
* a parameter space;
* a method for searching or sampling candidates;
* a cross-validation scheme; and
* a score function.


In [7]:
# examine the best model

# Single best score achieved across all params (min_samples_split)
print(grid.best_score_)

# Dictionary containing the parameters (min_samples_split) used to generate that score
print(grid.best_params_)

# Actual model object fit with those best parameters
# Shows default parameters that we did not specify
print(grid.best_estimator_)

0.9818181818181818
{'criterion': 'gini', 'min_samples_split': 4}
DecisionTreeClassifier(min_samples_split=4, random_state=0)
